# 66장 · 역할별 함수 분리와 호출 비용

역할을 함수로 나누면 한 함수의 결과가 다음 역할의 입력이 됩니다. 답변에 필요한 근거가 인계 과정에서 빠지는지와 추가 호출 수를 확인합니다.

과제 제출 질문을 한 에이전트가 처리한 경우와 근거 담당·답변 담당으로 나눈 경우를 비교합니다. 파이썬 함수로 인계한 질문과 문서 ID를 확인하고 두 실행 기록의 호출 수를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 근거 수집 함수와 답변 함수의 인계값

다음 역할이 문서 ID만 받으면 원문을 다시 찾아야 합니다. 질문, 근거 원문, 출처 ID를 함께 넘겨 답변의 수치나 위치를 확인할 수 있게 합니다. 여기서는 두 역할을 로컬 함수로 실행합니다.


### 셀 01 · 역할 간 근거 데이터 전달

evidence_role의 반환값에서 answer_role이 실제로 읽는 필드를 확인하세요.

answer_role은 text와 id로 문장을 만듭니다. question은 전달받아도 문장을 작성할 때 사용하지 않습니다.


In [1]:
def evidence_role(question):
    return {"question": question, "id": "course-guide-v1:p2",
            "text": "과제는 학습 관리 페이지의 과제함에 제출합니다."}

def answer_role(handoff):
    return f"{handoff['text']} [{handoff['id']}]"

handoff = evidence_role("과제는 어디에 내나요?")
print(answer_role(handoff))

과제는 학습 관리 페이지의 과제함에 제출합니다. [course-guide-v1:p2]


답변 끝의 p2는 근거 담당이 전달한 ID입니다. 문장을 만드는 쪽에서 새로 붙인 문서 번호가 아닙니다.

두 역할 사이에서 최종 문장만 전달하면 두 번째 역할은 p2의 어느 구절을 읽었는지 확인하기 어렵습니다. 원래 질문과 근거 ID를 함께 넘겨야 인계 전후의 요청이 같은지, 답변이 같은 근거를 사용했는지 대조할 수 있습니다.


### 두 역할 사이의 인계값

<p>evidence_role의 출력은 question·id·text를 담은 딕셔너리입니다. answer_role은 그중 text와 id를 읽어 한 문장으로 묶습니다. 이 셀의 역할은 일반 Python 함수입니다.</p>


### 과제 02 · 인계 시 질문 보존 검사

검색 담당이 받은 원 질문과 인계된 질문이 같은지 확인하세요.


In [2]:
question = '과제는 어디에 내나요?'
# handoff의 질문 필드를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>question = &#x27;과제는 어디에 내나요?&#x27;
print(question == handoff[&#x27;question&#x27;])</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>True입니다. 문서가 같더라도 질문이 바뀌면 두 방식의 결과를 같은 과제로 비교하기 어렵습니다.</p>
</details>


### 질문이 다음 역할까지 같은지 확인하기

<p>인계받은 근거가 다른 질문의 것이라면 형식이 맞아도 답변을 만들 수 없습니다. answer_role을 부르기 전에 handoff["question"]과 원래 질문을 대조합니다.</p>


<h3>역할 사이에 전달되는 딕셔너리</h3>

<p>evidence_role은 질문·문서 ID·원문을 한 객체에 넣고 answer_role은 그중 text와 id로 답을 만듭니다. 두 함수는 모델을 호출하지 않습니다. 뒤의 model_calls 2회와 4회는 셀에 따로 제공한 두 경로의 관찰 기록입니다.</p>


## 단일 경로와 역할 분리 경로 비교

셀 03의 딕셔너리 두 개에는 같은 질문 Q01의 대본 실행 기록이 들어 있습니다. 앞의 파이썬 함수 두 개가 이 횟수를 발생시킨 것은 아닙니다.


### 셀 03 · 역할별 호출 횟수 비교

single과 two_roles의 기록에서 모델 호출 횟수가 얼마나 달라졌는지 계산하세요.


In [3]:
single = {"model_calls": 2, "mandatory_retrieval_calls": 1}
two_roles = {"model_calls": 4, "mandatory_retrieval_calls": 1}
print(two_roles["model_calls"] - single["model_calls"])

2


출력 2는 제공 기록의 4회에서 2회를 뺀 값입니다. 필수 검색은 양쪽 모두 1회입니다. 호출당 토큰이나 단가는 기록에 없어 추가 비용은 계산할 수 없습니다.


### 같은 단위끼리 호출 수 비교하기

<p>두 관찰값 모두 동일 질문에서 필수 검색을 한 번 수행했습니다. model_calls의 차이와 mandatory_retrieval_calls의 차이를 따로 계산하면 무엇이 늘었는지 알 수 있습니다.</p>


### 역할이 늘 때 들인 비용

<p>기록에 적힌 모델 호출은 단일 경로에서 2회, 역할 분리 경로에서 4회입니다. 이 셀은 기록의 차이 2를 계산합니다. 두 모델을 새로 호출해 재측정하지는 않습니다.</p>


## 복습과 실습 점검

### 과제 04 · 인계한 근거가 새 질문에도 맞는지 확인하기

같은 함수에 주차 요금 질문을 넘기고 받은 근거 문장을 확인하세요. 두 번째 줄에서 주차 요금을 확인할 수 있는지 살펴보세요.


In [ ]:
# evidence_role에 "주차비는 얼마인가요?"를 전달해 원문을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>parking = evidence_role(&quot;주차비는 얼마인가요?&quot;)
print(parking[&quot;question&quot;])
print(parking[&quot;text&quot;])</code></pre>
<p>실행 결과</p><pre><code>주차비는 얼마인가요?
과제는 학습 관리 페이지의 과제함에 제출합니다.</code></pre>
<p>넘겨받은 질문은 정확하지만 근거는 과제 제출 안내입니다. evidence_role이 어떤 질문에도 같은 문서를 반환하기 때문입니다. 이 경우에는 근거를 새로 찾거나 빈 근거로 종료해야 하며, 같은 자료를 다른 역할에 한 번 더 넘겨도 주차비를 알 수 없습니다.</p>
</details>
